# Kaggle 电影推荐竞赛 - 主工作流程

### 简介
本 Notebook 用于 Kaggle 电影推荐竞赛的开发和实验。
流程包括：
1.  **环境配置**: 设置常量和路径。
2.  **数据处理**: 加载数据并构建用户-物品交互矩阵 (URM)。
3.  **本地验证**: 将数据分割为训练集和验证集。
4.  **模型训练与评估**: 训练基线模型并评估其性能。
5.  **实验区域**: 用于测试和比较新模型。
6.  **生成提交**: 使用最佳模型在全量数据上训练并生成提交文件。# Kaggle 电影推荐竞赛 - 主工作流程

## 1. 导入必要的库

In [2]:
# -*- coding: utf-8 -*-

import pandas as pd
import scipy.sparse as sps
import numpy as np
import os
import json
from tqdm import tqdm

from sklearn.model_selection import KFold

from RecSys_Course_AT_PoliMi.Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from RecSys_Course_AT_PoliMi.Evaluation.Evaluator import EvaluatorHoldout
from RecSys_Course_AT_PoliMi.Recommenders.Recommender_import_list import *
from Recommenders.BaseRecommender import BaseRecommender


## 2. 项目配置与常量

In [3]:
# 数据文件路径
DATA_FOLDER = 'dataset'
DATA_TRAIN_PATH = os.path.join(DATA_FOLDER, 'data_train.csv')
DATA_TARGET_USERS_PATH = os.path.join(DATA_FOLDER, 'data_target_users_test.csv')
OUTPUT_FOLDER = 'temp_output'
MODEL_FOLDER = 'model_output'
SUBMISSION_FOLDER = 'temp_output' # 提交文件的保存目录

# 随机种子，用于确保实验结果的可复现性
RANDOM_SEED = 1234

# 本地验证时，训练集所占的百分比
TRAIN_PERCENTAGE = 0.80

# 评估时使用的推荐列表长度 (cutoff)
EVALUATION_CUTOFF = 20

# 设置全局随机种子
np.random.seed(RANDOM_SEED)

print("项目配置加载完成.")

项目配置加载完成.


## 3. 辅助函数

### 3.1 加载CSV数据文件，并将其转换为CSR格式的稀疏矩阵.

In [4]:
def load_and_preprocess_data(file_path: str) -> sps.csr_matrix:
    """
    加载CSV数据文件，并将其转换为CSR格式的稀疏矩阵.
    """
    print("--- 正在加载和预处理数据... ---")
    df_train = pd.read_csv(file_path, dtype={'row': int, 'col': int})

    n_users = df_train['row'].max() + 1
    n_items = df_train['col'].max() + 1

    urm_all = sps.coo_matrix(
        ([1.0] * len(df_train['row']), (df_train['row'], df_train['col'])),
        shape=(n_users, n_items),
        dtype=float
    ).tocsr()
    print(f"数据加载完成. URM 维度: {urm_all.shape}")
    return urm_all


### 3.2 接收评估结果字典和模型名称，并以清晰的格式打印关键指标.

In [5]:
def print_results_formatted(results_df, model_name: str = "Model"):
    """
    接收评估结果 DataFrame 和模型名称，并以清晰的格式打印关键指标.

    Args:
        results_df (pd.DataFrame): 来自 Evaluator.evaluateRecommender() 的结果DataFrame.
        model_name (str): 要打印的模型名称.
    """
    # 检查 DataFrame 是否为空，或者指定的 cutoff 值是否作为索引存在
    if results_df.empty or EVALUATION_CUTOFF not in results_df.index:
        print(f"--- 在 cutoff={EVALUATION_CUTOFF} 处没有找到 '{model_name}' 的评估结果 ---")
        return

    # 使用 .loc[] 通过索引名来选取我们关心的那一行数据
    # 这会返回一个 Pandas Series，其行为非常像一个字典
    res_series = results_df.loc[EVALUATION_CUTOFF]

    print(f"--- 模型评估结果: {model_name} ---")
    print(f"--------------------------------------------------")
    # 现在在一个 Pandas Series 上使用 .get() 方法，这是安全且正确的
    print(f"{f'RECALL@{EVALUATION_CUTOFF}':<25}: {res_series.get('RECALL', -1):.4f}   <-- 竞赛官方指标")
    print(f"{f'PRECISION@{EVALUATION_CUTOFF}':<25}: {res_series.get('PRECISION', -1):.4f}")
    print(f"{f'MAP@{EVALUATION_CUTOFF}':<25}: {res_series.get('MAP', -1):.4f}")
    print(f"{f'HIT_RATE@{EVALUATION_CUTOFF}':<25}: {res_series.get('HIT_RATE', -1):.4f}")
    print(f"{f'ITEM_COVERAGE@{EVALUATION_CUTOFF}':<25}: {res_series.get('COVERAGE_ITEM', -1):.4f}")
    print(f"{f'AVG_POPULARITY@{EVALUATION_CUTOFF}':<25}: {res_series.get('AVERAGE_POPULARITY', -1):.4f}")
    print(f"--------------------------------------------------\n")

### 3.3 5-Kold 交叉验证

In [6]:
def evaluate_with_cross_validation(recommender_class, urm_all, k=5, recommender_params={}):
    """
    使用 K-Fold 交叉验证来评估一个推荐模型。

    Args:
        recommender_class: 要评估的推荐器类 (e.g., ItemKNNCFRecommender).
        urm_all (sps.csr_matrix): 全量的用户-物品交互矩阵.
        k (int): 折数 (e.g., 5).
        recommender_params (dict): 传递给推荐器 fit 方法的超参数.

    Returns:
        float: K 次评估的平均 RECALL@20 分数.
    """
    # KFold 会将数据分割成 k 个部分
    # 我们这里分割的是交互记录的索引
    kfold = KFold(n_splits=k, shuffle=True, random_state=RANDOM_SEED)

    # 存储每一折的 recall 分数
    recall_scores = []

    # 获取非零元素的行和列索引，代表了所有的交互
    rows, cols = urm_all.nonzero()

    print(f"--- 开始 {k}-Fold 交叉验证 ({recommender_class.RECOMMENDER_NAME}) ---")

    # fold_num 是从 1 开始的计数器
    for fold_num, (train_indices, validation_indices) in enumerate(kfold.split(rows), 1):
        print(f"--- 第 {fold_num}/{k} 折 ---")

        # 1. 根据索引创建训练集和验证集
        urm_train = sps.csr_matrix((np.ones(len(train_indices)), (rows[train_indices], cols[train_indices])), shape=urm_all.shape)
        urm_validation = sps.csr_matrix((np.ones(len(validation_indices)), (rows[validation_indices], cols[validation_indices])), shape=urm_all.shape)

        # 2. 初始化评估器
        evaluator = EvaluatorHoldout(urm_validation, cutoff_list=[EVALUATION_CUTOFF])

        # 3. 训练模型
        recommender = recommender_class(urm_train)
        recommender.fit(**recommender_params)

        # 4. 评估模型
        results_df, _ = evaluator.evaluateRecommender(recommender)

        # 5. 提取并存储 recall
        recall = results_df.loc[EVALUATION_CUTOFF].get('RECALL', 0.0)
        recall_scores.append(recall)

        print(f"第 {fold_num}/{k} 折 RECALL@{EVALUATION_CUTOFF}: {recall:.4f}")

    # 6. 计算并返回平均分
    mean_recall = np.mean(recall_scores)
    print(f"\n--- 交叉验证完成 ---")
    print(f"平均 RECALL@{EVALUATION_CUTOFF}: {mean_recall:.4f}\n")

    return mean_recall

### 3.4 读取模型

In [7]:
def load_best_model(recommender_class, urm_train, file_name, modelfile_path):
    """
    加载一个由超参数搜索脚本保存的最佳模型。
    """
    file_path = os.path.join(modelfile_path, file_name)

    print(f"--- 正在加载预训练模型: {file_name} ---")

    # 检查模型文件是否存在
    if not os.path.exists(file_path + ".zip"):
        print(f">>> 警告: 模型文件 '{file_path}.zip' 不存在!")
        print(">>> 请确保超参数优化已完成，并且文件名正确。")
        return None

    # 1. 初始化一个“空”的模型对象
    recommender_instance = recommender_class(urm_train)

    # 2. 调用 .load_model() 方法加载数据
    recommender_instance.load_model(folder_path=modelfile_path, file_name=file_name)

    print("模型加载成功！\n")
    return recommender_instance

## 4. 本地验证流程
本节将加载数据，并将其分割为训练集和验证集，为模型评估做准备。

In [8]:
# 加载数据
urm_all = load_and_preprocess_data(DATA_TRAIN_PATH)

# 分割数据用于本地验证
print("\n--- 正在分割数据用于本地验证... ---")
URM_train, URM_validation = split_train_in_two_percentage_global_sample(
    urm_all,
    train_percentage=TRAIN_PERCENTAGE
)
print("数据分割完成.")
print(f"训练集维度: {URM_train.shape}, 验证集维度: {URM_validation.shape}\n")


# 初始化评估器
evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[EVALUATION_CUTOFF])
print("评估器初始化完成.")

--- 正在加载和预处理数据... ---
数据加载完成. URM 维度: (27095, 6969)

--- 正在分割数据用于本地验证... ---
数据分割完成.
训练集维度: (27095, 6969), 验证集维度: (27095, 6969)

EvaluatorHoldout: Ignoring 37 ( 0.1%) Users that have less than 1 test interactions
评估器初始化完成.


## 5. 模型训练与实验
在这里训练、评估和比较不同的推荐模型。

#### 5a. 运行更稳健的评估方法: K-Fold 交叉验证

In [ ]:
# --- 使用交叉验证来评估你的 ItemKNNCF 模型 ---
itemknn_params = {'topK': 100, 'shrink': 10, 'similarity': 'cosine'}
# 运行交叉验证
avg_recall_itemknn = evaluate_with_cross_validation(ItemKNNCFRecommender, urm_all, k=5, recommender_params=itemknn_params)

#### 5b. 运行模型以获得输出

In [14]:
# ----- 1. 加载并评估 SLIMElasticNetRecommender (冠军模型) -----

modelfile_name = "MatrixFactorization_BPR_Cython_Recommender_best_model"
modelfile_path = "temp_output"
recommender_loaded = load_best_model(MatrixFactorization_BPR_Cython, URM_train, modelfile_name, modelfile_path)

if recommender_loaded:
    results_df, _ = evaluator_validation.evaluateRecommender(recommender_loaded)
    print_results_formatted(results_df, "Loaded")


--- 正在加载预训练模型: MatrixFactorization_BPR_Cython_Recommender_best_model ---
MatrixFactorization_BPR_Cython_Recommender: Loading model from file 'temp_outputMatrixFactorization_BPR_Cython_Recommender_best_model'
MatrixFactorization_BPR_Cython_Recommender: Loading complete
模型加载成功！

EvaluatorHoldout: Processed 27058 (100.0%) in 9.93 sec. Users per second: 2724
--- 模型评估结果: Loaded ---
--------------------------------------------------
RECALL@20                : 0.1722   <-- 竞赛官方指标
PRECISION@20             : 0.1381
MAP@20                   : 0.0547
HIT_RATE@20              : 0.8743
ITEM_COVERAGE@20         : 0.2051
AVG_POPULARITY@20        : 0.4098
--------------------------------------------------



### 5.1 基线模型: Item-based KNN CF

In [ ]:
print("--- 正在训练基线模型 (ItemKNNCF)... ---")
# 初始化模型实例
recommender_itemknn = ItemKNNCFRecommender(URM_train)

# 训练模型 (拟合相似度矩阵)
# 这些是模型的超参数，是后续优化的重点
recommender_itemknn.fit(topK=100, shrink=10, similarity='cosine')
print("模型训练完成.\n")

# 评估模型
results_dict_itemknn, _ = evaluator_validation.evaluateRecommender(recommender_itemknn)

# 打印格式化的结果
print_results_formatted(results_dict_itemknn, "ItemKNNCF Baseline")

### 5.2 实验模型: P3alpha (Code)

In [ ]:
print("--- 正在训练实验模型 (P3alpha)... ---")
# 初始化模型实例
recommender_p3alpha = P3alphaRecommender(URM_train)

# 训练模型
# P3alpha 在隐式反馈数据集上通常表现优异
recommender_p3alpha.fit(topK=100, alpha=0.8, implicit=True)
print("模型训练完成.\n")

# 评估模型
results_dict_p3alpha, _ = evaluator_validation.evaluateRecommender(recommender_p3alpha)

# 打印格式化的结果
print_results_formatted(results_dict_p3alpha, "P3alpha")

### 5.3 实验模型: SLIM BPR (Sparse LInear Methods with BPR)

In [ ]:
print("--- 正在训练实验模型 (SLIM BPR)... ---")
# 初始化模型实例
# 注意：SLIM BPR 是用 Cython 实现的，以获得更好的性能
recommender_slim_bpr = SLIM_BPR_Cython(URM_train)

# 训练模型
# SLIM BPR 是一个机器学习模型，它有 epochs, learning_rate 等参数
# 下面是一组常用的初始参数，是后续超参数优化的重点
recommender_slim_bpr.fit(
    epochs=100,          # 迭代次数
    topK=150,            # 邻域大小
    lambda_i=0.001,      # L2正则化系数 (items)
    lambda_j=0.001,      # L2正则化系数 (users)
    learning_rate=0.01
)
print("模型训练完成.\n")

# 评估模型
# 注意：我们将结果保存在一个新的变量中，以便后续比较和融合
results_df_slim_bpr, _ = evaluator_validation.evaluateRecommender(recommender_slim_bpr)

# 打印格式化的结果
print_results_formatted(results_df_slim_bpr, "SLIM BPR")

### 5.4 实验模型: IALS (Implicit Alternating Least Squares)

In [21]:
print("--- 正在训练实验模型 (IALS)... ---")
# 初始化模型实例
recommender_ials = IALSRecommender(URM_train)

# 训练模型
# IALS 也是一个迭代式的机器学习模型
# num_factors 是最重要的超参数之一，它决定了潜在空间的维度
recommender_ials.fit(
    num_factors = 58,
    epochs = 20,
    confidence_scaling = "log",
    alpha = 49.99999999999999,
    epsilon = 5.585081217734329,
    reg = 0.0007759360926311159
)
print("模型训练完成.\n")

# 评估模型
results_df_ials, _ = evaluator_validation.evaluateRecommender(recommender_ials)

# 打印格式化的结果
print_results_formatted(results_df_ials, "IALS")

--- 正在训练实验模型 (IALS)... ---
IALSRecommender: Epoch 1 of 20. Elapsed time 3.77 sec
IALSRecommender: Epoch 2 of 20. Elapsed time 7.56 sec
IALSRecommender: Epoch 3 of 20. Elapsed time 11.38 sec
IALSRecommender: Epoch 4 of 20. Elapsed time 15.09 sec
IALSRecommender: Epoch 5 of 20. Elapsed time 18.76 sec
IALSRecommender: Epoch 6 of 20. Elapsed time 22.52 sec
IALSRecommender: Epoch 7 of 20. Elapsed time 26.26 sec
IALSRecommender: Epoch 8 of 20. Elapsed time 29.98 sec
IALSRecommender: Epoch 9 of 20. Elapsed time 33.76 sec
IALSRecommender: Epoch 10 of 20. Elapsed time 37.54 sec
IALSRecommender: Epoch 11 of 20. Elapsed time 41.32 sec
IALSRecommender: Epoch 12 of 20. Elapsed time 45.05 sec
IALSRecommender: Epoch 13 of 20. Elapsed time 48.86 sec
IALSRecommender: Epoch 14 of 20. Elapsed time 52.78 sec
IALSRecommender: Epoch 15 of 20. Elapsed time 59.07 sec
IALSRecommender: Epoch 16 of 20. Elapsed time 1.09 min
IALSRecommender: Epoch 17 of 20. Elapsed time 1.17 min
IALSRecommender: Epoch 18 of 20. E

### 5.5 实验模型：SLIMElasticNetRecommender (使用局部最优参数)

In [ ]:
print("--- 正在使用最优参数训练冠军模型 (SLIMElasticNet)... ---")

# 初始化模型实例
recommender_slim_elastic = SLIMElasticNetRecommender(URM_train)

# 训练模型
# 使用我们在超参数搜索中找到的最佳参数
# 注意：即使有最优参数，这个模型的训练过程也可能需要较长时间 (几十分钟甚至更久)
recommender_slim_elastic.fit(
    topK=821,
    l1_ratio=0.019917913540297195, # 为了精确复现，我们使用完整的浮点数
    alpha=0.001
)
print("模型训练完成。\n")

# 评估模型
results_df_slim_elastic, _ = evaluator_validation.evaluateRecommender(recommender_slim_elastic)

# 打印格式化的结果
print_results_formatted(results_df_slim_elastic, "SLIMElasticNet (Optimized)")

## 6. 模型融合

### 6a 双模型融合

#### 6.1 定义融合推荐器

In [15]:
# --- 定义一个通用的混合推荐器类 ---
class HybridScoreRecommender(BaseRecommender):
    """
    一个通用的混合推荐器，通过加权平均多个模型的分数来进行推荐。
    """
    def __init__(self, urm_train, recommenders, weights):
        super(HybridScoreRecommender, self).__init__(urm_train)
        self.recommenders = recommenders
        self.weights = weights

    def _compute_item_score(self, user_id_array, items_to_compute=None):
        # 初始化一个零分矩阵
        final_scores = np.zeros((len(user_id_array), self.n_items), dtype=np.float32)

        for i, recommender in enumerate(self.recommenders):
            # 获取单个模型的分数
            scores = recommender._compute_item_score(user_id_array, items_to_compute)

            # 归一化分数
            scores = self._normalize_scores(scores)

            # 加权求和
            final_scores += self.weights[i] * scores

        return final_scores

    def _normalize_scores(self, scores):
        """对分数矩阵的每一行进行 Min-Max 归一化"""
        # 检查是否是稀疏矩阵，如果是，则转换为稠密数组
        if sps.issparse(scores):
            scores = scores.toarray()

        max_val = scores.max(axis=1, keepdims=True)
        min_val = scores.min(axis=1, keepdims=True)

        # 避免除以零
        denominator = max_val - min_val
        denominator[denominator == 0] = 1.0

        return (scores - min_val) / denominator

print("模型融合工具已准备就绪。")

模型融合工具已准备就绪。


#### 6.2 加载已经训练好的两个最佳模型

In [16]:
# --- 1. 加载模型 ---
# 定义模型所在的文件夹
COMBINE_MODEL_FOLDER = "temp_output"

print("--- 正在加载用于融合的基模型... ---")

# 加载 SLIMElasticNetRecommender
recommender_slim = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=URM_train, # 使用 URM_train 初始化
    file_name="SLIMElasticNetRecommender_best_model",
    modelfile_path=COMBINE_MODEL_FOLDER
)

# 加载 b_best_model
recommender_b = load_best_model(
    recommender_class=IALSRecommender,
    urm_train=URM_train, # 使用 URM_train 初始化
    file_name="IALSRecommender_best_model",
    modelfile_path=COMBINE_MODEL_FOLDER
)

# 确保两个模型都成功加载
if recommender_slim is None or recommender_b is None:
    print(">>> 错误：一个或多个基模型加载失败，无法继续进行融合。")
else:
    print("所有基模型均已成功加载！")

--- 正在加载用于融合的基模型... ---
--- 正在加载预训练模型: SLIMElasticNetRecommender_best_model ---
SLIMElasticNetRecommender: Loading model from file 'temp_outputSLIMElasticNetRecommender_best_model'
SLIMElasticNetRecommender: Loading complete
模型加载成功！

--- 正在加载预训练模型: IALSRecommender_best_model ---
IALSRecommender: Loading model from file 'temp_outputIALSRecommender_best_model'
IALSRecommender: Loading complete
模型加载成功！

所有基模型均已成功加载！


#### 6.3 网格搜索寻找最佳融合权重

In [ ]:
# --- 2. 网格搜索寻找最佳融合权重 ---

best_recall = 0
best_alpha = 0
alpha_values = np.arange(0.05, 1.0, 0.05) # 以0.05为步长，进行更精细的搜索

# 确保模型已成功加载
if recommender_slim and recommender_b:
    print("\n--- 开始在本地验证集上寻找最佳融合权重 alpha ---")

    # 将模型实例放入一个列表中
    recommender_list = [recommender_slim, recommender_b]

    for alpha in tqdm(alpha_values, desc="搜索 Alpha 权重"):
        # alpha 是第一个模型的权重, (1-alpha) 是第二个模型的权重
        weights = [alpha, 1 - alpha]

        # 创建混合推荐器实例
        hybrid_recommender = HybridScoreRecommender(URM_train, recommender_list, weights)

        # 在验证集上评估
        results_df, _ = evaluator_validation.evaluateRecommender(hybrid_recommender)
        current_recall = results_df.loc[EVALUATION_CUTOFF].get('RECALL', 0.0)

        if current_recall > best_recall:
            best_recall = current_recall
            best_alpha = alpha
            print(f"发现新的最佳 Recall: {best_recall:.5f} (当 alpha = {best_alpha:.2f})")

    print(f"\n--- 搜索完成！---")
    print(f"最佳 Alpha (SLIMElasticNet 的权重): {best_alpha:.2f}")
    print(f"RP3beta 的权重: {(1 - best_alpha):.2f}")
    print(f"融合后在本地验证集上的最佳 Recall@20: {best_recall:.5f}")

#### 6.4 使用最佳权重生成提交文件

In [24]:
# --- 1. 配置最佳权重和模型信息 ---
BEST_ALPHA = 0.70 # !!! 请确保将此值替换为您在网格搜索中找到的最佳 alpha !!!

MODEL_FOLDER = 'model_output' # 模型文件所在的文件夹

# --- 2. 加载在 *全量数据* 上训练好的模型 ---
print("\n--- 正在加载在全量数据上预训练的最终模型... ---")

# 加载 SLIMElasticNetRecommender
final_recommender_slim = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=urm_all, # 使用 urm_all 初始化
    file_name="5-1final_model_SLIMElasticNetRecommender-better",
    modelfile_path=MODEL_FOLDER
)

# 加载 b
final_recommender_b = load_best_model(
    recommender_class=IALSRecommender,
    urm_train=urm_all, # 使用 urm_all 初始化
    file_name="5-2final_model_IALSRecommender",
    modelfile_path=MODEL_FOLDER
)


# --- 3. 创建最终的混合推荐器并生成提交 ---
if final_recommender_slim and final_recommender_b:

    final_recommender_list = [final_recommender_slim, final_recommender_b]
    final_weights = [BEST_ALPHA, 1 - BEST_ALPHA]

    final_hybrid_recommender = HybridScoreRecommender(urm_all, final_recommender_list, final_weights)

    # --- 开始生成推荐 ---
    df_target_users = pd.read_csv(DATA_TARGET_USERS_PATH)
    target_user_ids = df_target_users['user_id'].values

    submission = []
    print(f"--- 正在为 {len(target_user_ids)} 名目标用户生成最终融合推荐... ---")
    for user_id in tqdm(target_user_ids):
        recommended_items = final_hybrid_recommender.recommend(
            user_id,
            cutoff=EVALUATION_CUTOFF,
            remove_seen_flag=True
        )
        submission.append((user_id, ' '.join(map(str, recommended_items))))

    # --- 保存提交文件 ---
    submission_filename = f"submission_FINAL_Hybrid_SLIM_RP3_alpha_{BEST_ALPHA:.2f}.csv"
    SUBMISSION_PATH = os.path.join(SUBMISSION_FOLDER, submission_filename) # SUBMISSION_FOLDER 在之前已定义

    df_submission = pd.DataFrame(submission, columns=['user_id', 'item_list'])
    df_submission.to_csv(SUBMISSION_PATH, index=False)

    print(f"\n--- 最终提交文件已成功生成! ---")
    print(f"文件保存在: {SUBMISSION_PATH}")
else:
    print("\n>>> 错误：一个或多个最终模型加载失败，无法生成提交文件。")


--- 正在加载在全量数据上预训练的最终模型... ---
--- 正在加载预训练模型: 5-1final_model_SLIMElasticNetRecommender-better ---
SLIMElasticNetRecommender: Loading model from file 'model_output5-1final_model_SLIMElasticNetRecommender-better'
SLIMElasticNetRecommender: Loading complete
模型加载成功！

--- 正在加载预训练模型: 5-2final_model_IALSRecommender ---
IALSRecommender: Loading model from file 'model_output5-2final_model_IALSRecommender'


  1%|          | 181/27095 [00:00<00:15, 1765.01it/s]

IALSRecommender: Loading complete
模型加载成功！

--- 正在为 27095 名目标用户生成最终融合推荐... ---


100%|██████████| 27095/27095 [00:13<00:00, 1986.80it/s]



--- 最终提交文件已成功生成! ---
文件保存在: temp_output\submission_FINAL_Hybrid_SLIM_RP3_alpha_0.70.csv


### 6b 三模型

#### 6.1 加载模型

In [25]:
# --- 1. 加载模型 ---

print("--- 正在加载用于三模型融合的基模型... ---")

# 加载 SLIMElasticNetRecommender
recommender_1 = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=URM_train,
    file_name="SLIMElasticNetRecommender_best_model",
    modelfile_path=COMBINE_MODEL_FOLDER
)

# 加载 RP3betaRecommender
recommender_2 = load_best_model(
    recommender_class=RP3betaRecommender,
    urm_train=URM_train,
    file_name="RP3betaRecommender_best_model",
    modelfile_path=COMBINE_MODEL_FOLDER
)

# 加载 IALSRecommender
recommender_3 = load_best_model(
    recommender_class=IALSRecommender,
    urm_train=URM_train,
    file_name="IALSRecommender_best_model",
    modelfile_path=COMBINE_MODEL_FOLDER
)

# 确保所有模型都成功加载
if not all([recommender_1, recommender_2, recommender_3]):
    print(">>> 错误：一个或多个基模型加载失败，无法继续进行三模型融合。")
else:
    print("所有基模型均已成功加载！准备进行权重搜索。")

--- 正在加载用于三模型融合的基模型... ---
--- 正在加载预训练模型: SLIMElasticNetRecommender_best_model ---
SLIMElasticNetRecommender: Loading model from file 'temp_outputSLIMElasticNetRecommender_best_model'
SLIMElasticNetRecommender: Loading complete
模型加载成功！

--- 正在加载预训练模型: RP3betaRecommender_best_model ---
RP3betaRecommender: Loading model from file 'temp_outputRP3betaRecommender_best_model'
RP3betaRecommender: Loading complete
模型加载成功！

--- 正在加载预训练模型: IALSRecommender_best_model ---
IALSRecommender: Loading model from file 'temp_outputIALSRecommender_best_model'
IALSRecommender: Loading complete
模型加载成功！

所有基模型均已成功加载！准备进行权重搜索。


#### 6.2 三模型融合的网格搜索

In [ ]:
best_recall = 0
best_weights = (0, 0, 0)
# 以0.1为步长进行搜索，以在合理时间内完成。如果想更精细，可以改为0.05
step = 0.1
weight_values = np.arange(0.5, 1.0 + step, step)

# 确保模型已成功加载
if all([recommender_1, recommender_2, recommender_3]):
    print("\n--- 开始在本地验证集上寻找最佳的三模型融合权重 ---")

    recommender_list = [recommender_1, recommender_2, recommender_3]

    # 创建一个进度条
    # 总的组合数可以通过一些数学计算得出，这里为了简单先预估一个大概的范围
    total_combinations = len(weight_values) ** 2
    pbar = tqdm(total=total_combinations, desc="搜索三模型权重")

    # w1 是 recommender_1 (SLIMElasticNet) 的权重
    for w1 in weight_values:
        # w2 是 recommender_2 (RP3beta) 的权重
        for w2 in weight_values:
            w3 = 1.0 - w1 - w2
            pbar.update(1) # 更新进度条

            # 确保所有权重都是非负的
            if w3 >= 0:
                weights = [w1, w2, w3]

                # 创建混合推荐器实例
                hybrid_recommender = HybridScoreRecommender(URM_train, recommender_list, weights)

                # 在验证集上评估
                results_df, _ = evaluator_validation.evaluateRecommender(hybrid_recommender)
                current_recall = results_df.loc[EVALUATION_CUTOFF].get('RECALL', 0.0)

                if current_recall > best_recall:
                    best_recall = current_recall
                    best_weights = tuple(weights)
                    # 实时打印出更好的结果
                    print(f"\n发现新的最佳 Recall: {best_recall:.5f} (权重: w1={best_weights[0]:.2f}, w2={best_weights[1]:.2f}, w3={best_weights[2]:.2f})")

    pbar.close() # 关闭进度条
    print(f"\n--- 搜索完成！---")
    print(f"最佳权重 (SLIMElasticNet, RP3beta, IALS): ({best_weights[0]:.2f}, {best_weights[1]:.2f}, {best_weights[2]:.2f})")
    print(f"融合后在本地验证集上的最佳 Recall@20: {best_recall:.5f}")

#### 6.3 三模型生成最终文件

In [27]:
# 权重的顺序必须与下面加载模型的顺序一致：(w_slim, w_rp3, w_ials)
BEST_WEIGHTS = (0.60, 0.10, 0.30)

MODEL_FOLDER = 'model_output' # 存放全量数据训练模型的文件夹


# --- 2. 加载在 *全量数据* 上训练好的三个模型 ---
print("\n--- 正在加载在全量数据上预训练的最终模型... ---")

# 加载 SLIMElasticNetRecommender (100% data version)
final_recommender_1 = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=urm_all, # 必须用 urm_all 初始化以正确过滤已看物品
    file_name="5-1final_model_SLIMElasticNetRecommender-better", # 替换为您保存的文件名
    modelfile_path=MODEL_FOLDER
)

# 加载 RP3betaRecommender (100% data version)
final_recommender_2 = load_best_model(
    recommender_class=RP3betaRecommender,
    urm_train=urm_all, # 必须用 urm_all 初始化
    file_name="3-1final_model_RP3betaRecommender", # 替换为您保存的文件名
    modelfile_path=MODEL_FOLDER
)

# 加载 IALSRecommender (100% data version)
final_recommender_3 = load_best_model(
    recommender_class=IALSRecommender,
    urm_train=urm_all, # 必须用 urm_all 初始化
    file_name="5-2final_model_IALSRecommender", # 替换为您保存的文件名
    modelfile_path=MODEL_FOLDER
)


# --- 3. 创建最终的混合推荐器并生成提交 ---
if all([final_recommender_1, final_recommender_2, final_recommender_3]):

    final_recommender_list = [final_recommender_1, final_recommender_2, final_recommender_3]

    # 确保权重数量与模型数量匹配
    assert len(final_recommender_list) == len(BEST_WEIGHTS), "模型数量与权重数量不匹配!"

    final_hybrid_recommender = HybridScoreRecommender(urm_all, final_recommender_list, BEST_WEIGHTS)

    # --- 开始生成推荐 ---
    df_target_users = pd.read_csv(DATA_TARGET_USERS_PATH)
    target_user_ids = df_target_users['user_id'].values

    submission = []
    print(f"--- 正在为 {len(target_user_ids)} 名目标用户生成最终的三模型融合推荐... ---")
    for user_id in tqdm(target_user_ids):
        recommended_items = final_hybrid_recommender.recommend(
            user_id,
            cutoff=EVALUATION_CUTOFF,
            remove_seen_flag=True
        )
        submission.append((user_id, ' '.join(map(str, recommended_items))))

    # --- 保存提交文件 ---
    # 创建一个能清晰描述此次提交的文件名
    submission_filename = f"submission_Hybrid_SLIM_RP3_IALS_w_{BEST_WEIGHTS[0]:.2f}_{BEST_WEIGHTS[1]:.2f}_{BEST_WEIGHTS[2]:.2f}.csv"
    SUBMISSION_PATH = os.path.join(SUBMISSION_FOLDER, submission_filename) # SUBMISSION_FOLDER 在之前已定义

    df_submission = pd.DataFrame(submission, columns=['user_id', 'item_list'])
    df_submission.to_csv(SUBMISSION_PATH, index=False)

    print(f"\n--- 最终提交文件已成功生成! ---")
    print(f"文件保存在: {SUBMISSION_PATH}")
else:
    print("\n>>> 错误：一个或多个最终模型加载失败，无法生成提交文件。")


--- 正在加载在全量数据上预训练的最终模型... ---
--- 正在加载预训练模型: 5-1final_model_SLIMElasticNetRecommender-better ---
SLIMElasticNetRecommender: Loading model from file 'model_output5-1final_model_SLIMElasticNetRecommender-better'
SLIMElasticNetRecommender: Loading complete
模型加载成功！

--- 正在加载预训练模型: 3-1final_model_RP3betaRecommender ---
RP3betaRecommender: Loading model from file 'model_output3-1final_model_RP3betaRecommender'
RP3betaRecommender: Loading complete
模型加载成功！

--- 正在加载预训练模型: 5-2final_model_IALSRecommender ---


  0%|          | 0/27095 [00:00<?, ?it/s]

IALSRecommender: Loading model from file 'model_output5-2final_model_IALSRecommender'
IALSRecommender: Loading complete
模型加载成功！

--- 正在为 27095 名目标用户生成最终的三模型融合推荐... ---


100%|██████████| 27095/27095 [00:26<00:00, 1041.85it/s]



--- 最终提交文件已成功生成! ---
文件保存在: temp_output\submission_Hybrid_SLIM_RP3_IALS_w_0.60_0.10_0.30.csv


## 7. 生成最终提交文件
当你通过本地验证找到了最佳模型和参数后，在这里使用 **全部训练数据** (`urm_all`) 进行训练，并为测试集用户生成推荐。

### 7a. 重新训练以生成提交文件

In [23]:
# --- 模型配置 ---
# 取消注释你想要使用的模型，并确保参数是最佳的

# # 配置
model_config = {
    "class": IALSRecommender,
    "params": {
        "num_factors": 58,
        "epochs": 30,
        "confidence_scaling": "log",
        "alpha": 49.99999999999999,
        "epsilon": 5.585081217734329,
        "reg": 0.0007759360926311159
    }
}


# 自动生成与模型相关的文件名，便于维护
model_name = model_config['class'].RECOMMENDER_NAME
submission_filename = f"submission_{model_name}.csv"
model_filename = f"final_model_{model_name}"
params_filename = f"final_params_{model_name}.json"

SUBMISSION_PATH = os.path.join(SUBMISSION_FOLDER, submission_filename)
MODEL_SAVE_PATH = os.path.join(MODEL_FOLDER, model_filename)
PARAMS_SAVE_PATH = os.path.join(MODEL_FOLDER, params_filename)


# -----------------------------------------------------------------------------
# STEP 2: 执行生成流程 (通常无需修改以下代码)
# -----------------------------------------------------------------------------

# 确保输出和模型文件夹存在
for folder in [MODEL_FOLDER, SUBMISSION_FOLDER]:
    if not os.path.exists(folder):
        os.makedirs(folder)
        print(f"文件夹 '{folder}' 已创建。")

# 1. 确定最终模型类和参数
final_model_class = model_config["class"]
final_model_params = model_config["params"]

# 2. 在 *全部* 数据上训练最终模型
print(f"--- 正在使用模型 '{model_name}' 在全量数据上进行训练... ---")
# 警告：对于 SLIMElasticNet 等模型，这一步会非常耗时!
final_recommender = final_model_class(urm_all)
final_recommender.fit(**final_model_params)
print("最终模型训练完成。\n")


# 3. ***** 新增功能: 保存模型和参数 *****
print(f"--- 正在保存已训练的模型和参数... ---")
# 保存模型对象
final_recommender.save_model(folder_path=MODEL_FOLDER, file_name=model_filename)
# 保存参数字典为 JSON 文件
with open(PARAMS_SAVE_PATH, 'w') as f:
    json.dump(final_model_params, f, indent=4)
print(f"模型已保存至: {MODEL_SAVE_PATH}.zip")
print(f"参数已保存至: {PARAMS_SAVE_PATH}\n")


# 4. 加载需要预测的目标用户
df_target_users = pd.read_csv(DATA_TARGET_USERS_PATH)
target_user_ids = df_target_users['user_id'].values

# 5. 生成推荐并保存到文件
print(f"--- 正在为 {len(target_user_ids)} 名目标用户生成推荐... ---")
submission = []
for user_id in tqdm(target_user_ids):
    recommended_items = final_recommender.recommend(
        user_id,
        cutoff=EVALUATION_CUTOFF,
        remove_seen_flag=True
    )
    submission.append((user_id, ' '.join(map(str, recommended_items))))

# 6. 创建DataFrame并保存为CSV
df_submission = pd.DataFrame(submission, columns=['user_id', 'item_list'])
df_submission.to_csv(SUBMISSION_PATH, index=False)

print(f"\n--- 提交文件已成功生成! ---")
print(f"文件保存在: {SUBMISSION_PATH}")
print("\n文件预览:")
print(df_submission.head())

--- 正在使用模型 'IALSRecommender' 在全量数据上进行训练... ---
IALSRecommender: Epoch 1 of 30. Elapsed time 4.42 sec
IALSRecommender: Epoch 2 of 30. Elapsed time 9.44 sec
IALSRecommender: Epoch 3 of 30. Elapsed time 14.34 sec
IALSRecommender: Epoch 4 of 30. Elapsed time 19.36 sec
IALSRecommender: Epoch 5 of 30. Elapsed time 24.27 sec
IALSRecommender: Epoch 6 of 30. Elapsed time 30.02 sec
IALSRecommender: Epoch 7 of 30. Elapsed time 35.45 sec
IALSRecommender: Epoch 8 of 30. Elapsed time 40.30 sec
IALSRecommender: Epoch 9 of 30. Elapsed time 45.70 sec
IALSRecommender: Epoch 10 of 30. Elapsed time 51.02 sec
IALSRecommender: Epoch 11 of 30. Elapsed time 56.05 sec
IALSRecommender: Epoch 12 of 30. Elapsed time 1.02 min
IALSRecommender: Epoch 13 of 30. Elapsed time 1.10 min
IALSRecommender: Epoch 14 of 30. Elapsed time 1.18 min
IALSRecommender: Epoch 15 of 30. Elapsed time 1.26 min
IALSRecommender: Epoch 16 of 30. Elapsed time 1.34 min
IALSRecommender: Epoch 17 of 30. Elapsed time 1.44 min
IALSRecommender: E

  1%|          | 266/27095 [00:00<00:10, 2636.00it/s]

IALSRecommender: Saving complete
模型已保存至: model_output\final_model_IALSRecommender.zip
参数已保存至: model_output\final_params_IALSRecommender.json

--- 正在为 27095 名目标用户生成推荐... ---


100%|██████████| 27095/27095 [00:08<00:00, 3197.46it/s]


--- 提交文件已成功生成! ---
文件保存在: temp_output\submission_IALSRecommender.csv

文件预览:
   user_id                                          item_list
0        0  2399 5532 2628 5609 6411 4312 457 868 1252 360...
1        1  6810 2295 4936 2907 3460 1949 403 2509 5630 28...
2        2  2557 4264 5640 5323 3285 4841 5390 1539 4877 6...
3        3  1554 6771 3713 1974 2653 5252 1407 2167 6846 4...
4        4  2472 4373 5069 5847 3512 4532 5460 4835 6684 2...


### 7b. 读取模型以获得提交文件

In [ ]:
# --- STEP 1: 加载预训练的最佳模型 ---

# 使用我们之前编写的 load_best_model 辅助函数
# 注意：这里传入的是 urm_all，因为 recommend 函数需要访问完整的交互历史来移除已看过的物品
# 而模型本身是在 URM_train 上训练的，这些信息都保存在了模型文件中。
# 实际上，初始化时传入哪个 URM 并不影响加载，但传入 urm_all 在逻辑上更清晰。
final_recommender = load_best_model(SLIMElasticNetRecommender, urm_all)


if final_recommender:
    # --- STEP 2: 执行生成流程 ---

    # 1. 加载需要预测的目标用户
    df_target_users = pd.read_csv(DATA_TARGET_USERS_PATH)
    target_user_ids = df_target_users['user_id'].values

    # 2. 生成推荐并保存到文件
    submission = []
    # 为了让输出更美观，我们可以使用 tqdm 来显示进度条
    from tqdm import tqdm

    print(f"--- 正在为 {len(target_user_ids)} 名目标用户生成推荐... ---")
    for user_id in tqdm(target_user_ids):
        recommended_items = final_recommender.recommend(
            user_id,
            cutoff=EVALUATION_CUTOFF,
            remove_seen_flag=True
        )
        submission.append((user_id, ' '.join(map(str, recommended_items))))

    # 3. 创建DataFrame并保存为CSV
    submission_filename = f"submission_{final_recommender.RECOMMENDER_NAME}_optimized.csv"
    SUBMISSION_PATH = os.path.join(OUTPUT_FOLDER, submission_filename)

    df_submission = pd.DataFrame(submission, columns=['user_id', 'item_list'])
    df_submission.to_csv(SUBMISSION_PATH, index=False)

    print(f"\n--- 提交文件已成功生成! ---")
    print(f"文件保存在: {SUBMISSION_PATH}")
    print("\n文件预览:")
    print(df_submission.head())

else:
    print(">>> 模型加载失败，无法生成提交文件。请检查文件路径和名称。")